In [23]:
import re
import time
from urllib.parse import urljoin, urlsplit, urlunsplit

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from datetime import datetime
from zoneinfo import ZoneInfo

import html

In [24]:
BASE_URL = "https://www.inven.co.kr"
BOARD_URL = "https://www.inven.co.kr/board/maple/2304?my=con"
START_PAGE = 1
END_PAGE = 10
REQUEST_DELAY = 1.0

In [25]:

# 세션 설정
session = requests.Session()

session.headers.update(
    {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/139.0 Safari/537.36"
        ),
        "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
    }
)

retry = Retry(
    total=3,
    connect=3,
    read=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods={"GET", "POST"},
)

adapter = HTTPAdapter(max_retries=retry)

session.mount("https://", adapter)
session.mount("http://", adapter)

# 게시글 저장용 리스트와 중복 방지용 세트
all_posts = []
visited_urls = set()

In [20]:
COMMENT_URL = "https://www.inven.co.kr/common/board/comment.json.php"
BOARD_CODE = "2304"

def fetch_comments(session, article_url):
    article_code = urlsplit(article_url).path.rstrip("/").split("/")[-1]

    offset = 0
    page_size = 100

    while True:
        payload = {
            "comeidx": BOARD_CODE,
            "articlecode": article_code,
            "sortorder": "date",
            "act": "list",
            "out": "json",
            "replynick": "",
            "replyidx": 0,
            "uploadurl": "",
            "imageposition": "",
            "videoloading": "lazy",
        }

        if offset > 0:
            payload["titles"] = offset

        response = session.post(
            COMMENT_URL,
            params={
                "dummy": int(time.time() * 1000)
            },
            data=payload,
            headers={
                "Referer": article_url,
                "X-Requested-With": "XMLHttpRequest",
            },
            timeout=15,
        )

        response.raise_for_status()
        data = response.json()
        comments = [] 

        # 댓글 추출
        for comment_group in data.get("commentlist", []):
            for comment in comment_group.get("list", []):

                attr = comment.get("__attr__", {})

                # HTML entity 복원
                content = comment.get("o_comment", "")

                content = html.unescape(
                    html.unescape(content)
                ).replace("\xa0", " ")

                comments.append({
                    "comment_id": attr.get("cmtidx"),
                    "parent_id": attr.get("cmtpidx"),
                    "author": comment.get("o_name"),
                    "created_at": comment.get("o_date"),
                    "content": comment.get("o_comment"),
                    "recommend": comment.get("o_recommend"),
                    "not_recommend": comment.get("o_notrecommend"),
                })

        total_count = int(data.get("cmtcount", 0))

        if len(comments) >= total_count:
            break

        offset += page_size

    return comments

In [26]:
# 페이지별 수집
for page in range(START_PAGE, END_PAGE + 1):
    print("\n" + "=" * 60)
    print(f"{page}페이지 수집 시작")
    print("=" * 60)

    try:
        # 게시판 목록 페이지 요청
        response = session.get(
            BOARD_URL,
            params={"my":"con", "p": page},
            timeout=15
        )
        response.raise_for_status()

        if response.encoding is None or response.encoding.lower() == "iso-8859-1":
            response.encoding = response.apparent_encoding

        soup = BeautifulSoup(response.text, "html.parser")

        # 게시글 링크 추출
        articles = soup.select(".text-wrap .subject-link")
        print(f"[페이지 {page}] 게시글 링크 후보: {len(articles)}개")

        post_urls = []

        for article in articles:
            href = article.get("href")

            if not href:
                continue

            url = urljoin(BASE_URL, href)
            parsed = urlsplit(url)

            if not re.fullmatch(r"/board/maple/2304/\d+", parsed.path):
                continue

            clean_url = urlunsplit(
                (
                    parsed.scheme,
                    parsed.netloc,
                    parsed.path,
                    "",
                    ""
                )
            )

            post_urls.append(clean_url)

        post_urls = list(dict.fromkeys(post_urls))
        print(f"수집할 게시글: {len(post_urls)}개")

    except requests.RequestException as exc:
        print(f"[페이지 요청 실패] page={page}")
        print(exc)
        continue


    # 게시글 상세 내용 수집
    for index, url in enumerate(post_urls, start=1):
        if url in visited_urls:
            continue

        try:
            response = session.get(url, timeout=15)
            response.raise_for_status()

            if response.encoding is None or response.encoding.lower() == "iso-8859-1":
                response.encoding = response.apparent_encoding

            soup = BeautifulSoup(response.text, "html.parser")

            title_element = soup.select_one(".articleTitle")
            author_element = soup.select_one(".nickname")
            date_element = soup.select_one(".articleDate")
            content_element = soup.select_one(".contentBody")
            category_element = soup.select_one(".articleCategory")

            if title_element is None:
                raise ValueError(f"title을 찾지 못했습니다: {url}")

            if content_element is None:
                raise ValueError(f"content를 찾지 못했습니다: {url}")

            # ==========================================
            # 기존 게시글 정보 처리
            # ==========================================

            info_text = ""

            if date_element:
                node = date_element

                while node and getattr(node, "name", None) != "body":
                    text = node.get_text(" ", strip=True)

                    if "조회:" in text and "추천:" in text:
                        info_text = text
                        break

                    node = node.parent

            if category_element:
                category = (
                    category_element
                    .get_text(" ", strip=True)
                    .strip()
                    .strip("[]")
                    .strip()
                )
            else:
                category_match = re.search(r"\[([^\]]+)\]", info_text)

                category = (
                    category_match.group(1).strip()
                    if category_match
                    else None
                )

            # 4. views
            views_match = re.search(
                r"조회:\s*([\d,]+)",
                info_text
            )

            views = (
                int(views_match.group(1).replace(",", ""))
                if views_match
                else None
            )


            # 5. likes
            likes_match = re.search(
                r"추천:\s*([\d,]+)",
                info_text
            )

            likes = (
                int(likes_match.group(1).replace(",", ""))
                if likes_match
                else None
            )

            # 6. 결과 저장
            post = {
                "url": url,
                "category": category,
                "title": title_element.get_text(" ", strip=True),

                "author": (
                    author_element.get_text(" ", strip=True)
                    if author_element
                    else None
                ),

                "created_at": (
                    date_element.get_text(" ", strip=True)
                    if date_element
                    else None
                ),

                "views": views,
                "likes": likes,

                "content": content_element.get_text(
                    "\n",
                    strip=True
                ),
            }

            all_posts.append(post)
            visited_urls.add(url)

            print(
                f"[{page}페이지 {index}/{len(post_urls)}] "
                f"{post['title']}"
            )

        except requests.RequestException as exc:
            print(f"[HTTP 오류] {url}")
            print(exc)

        except Exception as exc:
            print(f"[파싱 오류] {url}")
            print(exc)

        time.sleep(REQUEST_DELAY)

    time.sleep(REQUEST_DELAY)


1페이지 수집 시작
[페이지 1] 게시글 링크 후보: 30개
수집할 게시글: 30개
[1페이지 1/30] 뉴비에서 메잘알까지 1편 - 쿨타임 시스템
[1페이지 2/30] 메이플M 렌 250 육성 이벤트 공략 (무과금)
[1페이지 3/30] [울티마 스쿼드] 스테이지 권장레벨, 잠재옵션표, 스킬퍼뎀, 장비 리스트 및 능력치 공유
[1페이지 4/30] 울티마 스쿼드 장비 / 잠재 정보
[1페이지 5/30] 울티마 스쿼드 정보들 (테섭 기준)
[1페이지 6/30] 운빨편차를 줄이는 메이린 최소컷 공략 꿀팁
[1페이지 7/30] 아이템 버닝 대체 5초뚝 템셋 가이드(심화, 장문, 스압)
[1페이지 8/30] 아이템 버닝 대체 템셋에 대하여(스압, 장문)
[1페이지 9/30] 스압) 쓰잘데기 없는 보스 TMI
[1페이지 10/30] 모험으로 알아온 메이린 개꿀팁 (일요일에 재획안하고 메이린 최소컷 깎으면서)
[1페이지 11/30] 에테리온 아티팩트 효율 분석 ( 일퀘 vs 몬파 vs 에픽던전 )
[1페이지 12/30] 레테 극딜순서 다시 짜왔음
[1페이지 13/30] 야누스 30렙 제자리 사냥터 모음 (세르니움~아르테리아)
[1페이지 14/30] 챌섭 레테 제자리 사냥터 모음 (세르니움~아르테리아)
[1페이지 15/30] 챌린저스 육성 가이드 공유(구글 시트)
[1페이지 16/30] 여름 이벤트 재화 획득량 정리 + 패스 효율
[1페이지 17/30] 챌린저 달려면 어디까지 잡아야하나요?
[1페이지 18/30] 챌섭 직업 정하다가 정리해본 전직업 유틸 모음집
[1페이지 19/30] 챌섭 직업선택 가이드 (수정 6/16)
[1페이지 20/30] 루시드 히든미션(꿈의미로) / 5월 업데이트 요약+@
[1페이지 21/30] 헬레나 나이트메어 작은 팁
[1페이지 22/30] 챌린저스 대비 캐릭터 선택 가이드
[1페이지 23/30] [23주년] 봄빛 풍경, 단풍빛 추억 공략
[1페이지 24/30] MVP리조트 슬슬 저축해놓으세요
[1페이지 25/30] 루시드) 이번주 헬레나돌면

In [28]:
columns = [
    "url",
    "category",
    "title",
    "author",
    "created_at",
    "views",
    "likes",
    "content",
]

df = pd.DataFrame(
    all_posts,
    columns=columns
)

output_path = "../../../data/raw/maple_inven_rag_원본(1~10p).csv"

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"\n총 수집 게시글: {len(df)}개")
print(f"CSV 파일 저장 완료: {output_path}")

df.head()


총 수집 게시글: 300개
CSV 파일 저장 완료: ../../../data/raw/maple_inven_rag_원본(1~10p).csv


,url,category,title,author,created_at,views,likes,content
0,https://www.inven.co.kr/board/maple/2304/48082,실험,뉴비에서 메잘알까지 1편 - 쿨타임 시스템,NaN,2026-08-03 12:43,12988,18,메이플의 수많은 시스템들 중 뉴비가 접하기 쉽지 않거나 정보가 파편화 되어있어 알기...
1,https://www.inven.co.kr/board/maple/2304/48066,메이플M,메이플M 렌 250 육성 이벤트 공략 (무과금),NaN,2026-08-01 15:52,37151,12,00:11\n1~203 튜토리얼\n00:39\n203~\n01:05\n메뉴 및 스킬...
2,https://www.inven.co.kr/board/maple/2304/48012,기타,"[울티마 스쿼드] 스테이지 권장레벨, 잠재옵션표, 스킬퍼뎀, 장비 리스트 및 능력치 공유",없는이용자,2026-07-26 11:56,318931,29,더 많은 메이플 관련 정보는\n너의 공격력 마력\nhttps://maple.dwje...
3,https://www.inven.co.kr/board/maple/2304/47984,아이템,울티마 스쿼드 장비 / 잠재 정보,NaN,2026-07-24 14:21,96343,13,"수정사항\n-피격시 5% 확률로 데미지의 10% 무시 표기 수정\n-4단계 무기,방..."
4,https://www.inven.co.kr/board/maple/2304/47971,사냥,울티마 스쿼드 정보들 (테섭 기준),NaN,2026-07-23 07:41,339767,37,0. 에스페시아 상자\n최대한 맵 밀어놓고 사용하기\n1. 메이플 끄고 있어도 보상...


In [ ]:
# tip 게시판 정제
df = pd.read_csv("../../../data/raw/maple_inven_rag_원본(1~10p).csv")

# print(df.shape)
# display(df.head(5))
# display(df.info())

df_cp = df.copy()

df_cp = df_cp.drop(columns=['author'])
display(df_cp.info())



<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   url         300 non-null    str  
 1   category    300 non-null    str  
 2   title       300 non-null    str  
 3   created_at  300 non-null    str  
 4   views       300 non-null    int64
 5   likes       300 non-null    int64
 6   content     299 non-null    str  
dtypes: int64(2), str(5)
memory usage: 16.5 KB


None